In [1]:
import subprocess
import time
import os

# 1. Install required system dependencies (zstd)
!apt-get update -y > /dev/null 2>&1
!apt-get install zstd -y > /dev/null 2>&1

# 2. Install Ollama binaries
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Initialize the server as a background daemon
print("[SYSTEM] Booting Ollama server...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5) # Allow port 11434 to bind

# 4. Pull the instruction-tuned Llama-3 8B parameters
!ollama pull llama3
print("[SYSTEM] Model weights loaded into VRAM.")

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%###########################################   99.7%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
[SYSTEM] Booting Ollama server...
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 6a0746a1ec1a:   0% ▕                  ▏ 458 KB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   1% ▕                  ▏  50 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   3% ▕                  ▏ 142 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a: 

In [2]:
import json
import requests
import os

# Define Kaggle-specific I/O paths
INPUT_JSON = "/kaggle/input/datasets/namanatgoel/diverse-rap-corpus-500/diverse_rap_corpus_500.json" # Adjust directory name
OUTPUT_TXT = "/kaggle/working/literary_rap_conditioned.txt"
OLLAMA_URL = "http://localhost:11434/api/generate"

def analyze_lyrics_local(lyrics_text):
    prompt = f"""
    Analyze the following rap lyrics. Output a strict JSON object with exactly two keys: "mood" and "genre".
    Example: {{"mood": "Aggressive", "genre": "Hardcore Rap"}}
    Do not output markdown, explanation, or extra text.
    
    Lyrics: {lyrics_text[:1000]}
    """
    
    payload = {
        "model": "llama3",
        "prompt": prompt,
        "stream": False,
        "format": "json" 
    }
    
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=60)
        response.raise_for_status()
        return json.loads(response.json().get("response", "{}"))
    except Exception as e:
        print(f"[INFERENCE ERROR] {e}")
        return {"mood": "Unknown", "genre": "Rap"}

def execute_local_pipeline():
    print(f"[INITIALIZING] Parsing {INPUT_JSON}")
    with open(INPUT_JSON, 'r', encoding='utf-8') as f:
        corpus = json.load(f)
        
    print(f"[PROCESSING] Commencing tagging for {len(corpus)} tracks. Hardware constraint: Unlocked.")
    
    with open(OUTPUT_TXT, 'a', encoding='utf-8') as out_file:
        for idx, song in enumerate(corpus):
            if idx % 50 == 0:
                print(f"  → Processed: {idx}/{len(corpus)}")
                
            tags = analyze_lyrics_local(song['lyrics'])
            mood = tags.get('mood', 'Unknown')
            genre = tags.get('genre', 'Rap')
            
            formatted_entry = (
                f"<|startoftext|>\n"
                f"<|mood|> {mood} <|genre|> {genre}\n"
                f"<|bars|>\n"
                f"{song['lyrics']}\n"
                f"<|endoftext|>\n\n"
            )
            out_file.write(formatted_entry)

    print(f"[COMPLETE] Conditioned dataset compiled at: {OUTPUT_TXT}")

if __name__ == "__main__":
    if not os.path.exists(INPUT_JSON):
        print(f"[FATAL] Input file not found: {INPUT_JSON}. Verify Kaggle upload path.")
    else:
        execute_local_pipeline()

[INITIALIZING] Parsing /kaggle/input/datasets/namanatgoel/diverse-rap-corpus-500/diverse_rap_corpus_500.json
[PROCESSING] Commencing tagging for 500 tracks. Hardware constraint: Unlocked.
  → Processed: 0/500
[INFERENCE ERROR] HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=60)
  → Processed: 50/500
  → Processed: 100/500
  → Processed: 150/500
  → Processed: 200/500
  → Processed: 250/500
  → Processed: 300/500
  → Processed: 350/500
  → Processed: 400/500
  → Processed: 450/500
[COMPLETE] Conditioned dataset compiled at: /kaggle/working/literary_rap_conditioned.txt
